In [70]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/age-prediction-summer-analytics1/Train_Data.csv
/kaggle/input/age-prediction-summer-analytics1/Test_Data.csv


Import Libraries

In [71]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

Load Data

In [72]:

train_df = pd.read_csv("/kaggle/input/age-prediction-summer-analytics1/Train_Data.csv")
test_df = pd.read_csv("/kaggle/input/age-prediction-summer-analytics1/Test_Data.csv")

train_df.head()

print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())


Train columns: ['SEQN', 'RIAGENDR', 'PAQ605', 'BMXBMI', 'LBXGLU', 'DIQ010', 'LBXGLT', 'LBXIN', 'age_group']
Test columns: ['SEQN', 'RIAGENDR', 'PAQ605', 'BMXBMI', 'LBXGLU', 'DIQ010', 'LBXGLT', 'LBXIN']


Preprocessing

In [73]:

train_df.drop(columns=["SEQN"], inplace=True)
test_ids = test_df["SEQN"]
test_df.drop(columns=["SEQN"], inplace=True)


X = train_df.drop(columns=["age_group"])
y = train_df["age_group"]


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test_df)

print("Missing values in train data:\n", train_df.isnull().sum())


Missing values in train data:
 RIAGENDR     18
PAQ605       13
BMXBMI       18
LBXGLU       13
DIQ010       18
LBXGLT       11
LBXIN         9
age_group    14
dtype: int64


Train-Validation Split

In [74]:
train_df = train_df.dropna(subset=['age_group'])

train_df = train_df.fillna(train_df.mean(numeric_only=True))

X = train_df.drop("age_group", axis=1)
y = train_df["age_group"]

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Model Training

In [75]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train_encoded)

val_preds = model.predict(X_val)

print("Validation F1 Score:", f1_score(y_val_encoded, val_preds))

val_preds_decoded = le.inverse_transform(val_preds)
y_val_decoded = le.inverse_transform(y_val_encoded)

print("\nClassification Report:\n", classification_report(y_val_decoded, val_preds_decoded))

Validation F1 Score: 0.12658227848101264

Classification Report:
               precision    recall  f1-score   support

       Adult       0.85      0.97      0.90       328
      Senior       0.31      0.08      0.13        63

    accuracy                           0.82       391
   macro avg       0.58      0.52      0.51       391
weighted avg       0.76      0.82      0.78       391



Predict on Test Set

In [76]:
test_scaled = pd.DataFrame(test_scaled)

test_scaled.fillna(test_scaled.mean(), inplace=True)

test_preds = model.predict(test_scaled)


Submission File

In [77]:
submission = pd.DataFrame({
    "SEQN": test_ids,
    "age_group": test_preds
})

submission = submission.replace([np.inf, -np.inf], np.nan)
submission = submission.dropna()

submission["SEQN"] = submission["SEQN"].astype(int)
submission["age_group"] = submission["age_group"].astype(int)

submission.to_csv("submission.csv", index=False)

submission.head()


,SEQN,age_group
0,77017,0
1,75580,0
2,73820,0
3,80489,0
4,82047,0
